In [1]:
!pip install -q sentence-transformers faiss-cpu pandas scikit-learn streamlit

In [2]:
import pandas as pd

data = {
    "ticket_id": [1,2,3,4,5,6,7,8],
    "raw_text": [
        "Hi, I was charged twice for my subscription this month. Can someone help me get a refund?",
        "The dashboard is loading very slowly since yesterday. Is there an outage?",
        "I would love a dark mode option in the mobile app.",
        "My invoice still shows the old address even though I updated it last week.",
        "I can't log in. It says my password is wrong but I'm sure it's correct.",
        "Do you support SSO with Okta?",
        "The export to CSV is broken. It downloads an empty file.",
        "Just wanted to say the new reporting feature is really helpful. Thanks!"
    ]
}

df = pd.DataFrame(data)
print(df.head())

   ticket_id                                           raw_text
0          1  Hi, I was charged twice for my subscription th...
1          2  The dashboard is loading very slowly since yes...
2          3  I would love a dark mode option in the mobile ...
3          4  My invoice still shows the old address even th...
4          5  I can't log in. It says my password is wrong b...


In [3]:
import json
import re

CATEGORIES = ["billing", "bug", "feature_request", "other"]
URGENCY = ["low", "medium", "high"]

def simple_extract(text: str) -> dict:
    text_lower = text.lower()

    # Very simple rule-based starting point (we will improve it)
    if any(word in text_lower for word in ["charge", "refund", "invoice", "payment", "billed"]):
        category = "billing"
    elif any(word in text_lower for word in ["slow", "broken", "error", "can't log", "outage", "bug"]):
        category = "bug"
    elif any(word in text_lower for word in ["would love", "feature", "add", "option", "request"]):
        category = "feature_request"
    else:
        category = "other"

    if any(word in text_lower for word in ["refund", "can't log", "outage", "broken", "twice"]):
        urgency = "high"
    elif any(word in text_lower for word in ["slow", "invoice", "address"]):
        urgency = "medium"
    else:
        urgency = "low"

    summary = text.strip()[:120] + ("..." if len(text) > 120 else "")

    next_action = {
        "billing": "Review billing records and respond within 4 hours",
        "bug": "Create engineering ticket and notify on-call if high urgency",
        "feature_request": "Log in product feedback tracker",
        "other": "Route to general support queue"
    }[category]

    return {
        "category": category,
        "urgency": urgency,
        "summary": summary,
        "recommended_next_action": next_action
    }

# Test it
print(simple_extract(df.loc[0, "raw_text"]))

{'category': 'billing', 'urgency': 'high', 'summary': 'Hi, I was charged twice for my subscription this month. Can someone help me get a refund?', 'recommended_next_action': 'Review billing records and respond within 4 hours'}


In [4]:
def validate_output(data: dict) -> bool:
    if data.get("category") not in CATEGORIES:
        return False
    if data.get("urgency") not in URGENCY:
        return False
    if not data.get("summary") or not data.get("recommended_next_action"):
        return False
    return True

# Test
result = simple_extract(df.loc[0, "raw_text"])
print("Valid:", validate_output(result))

Valid: True


In [5]:
test_data = [
    {"text": "Hi, I was charged twice for my subscription this month. Can someone help me get a refund?", "expected_category": "billing", "expected_urgency": "high"},
    {"text": "The dashboard is loading very slowly since yesterday. Is there an outage?", "expected_category": "bug", "expected_urgency": "high"},
    {"text": "I would love a dark mode option in the mobile app.", "expected_category": "feature_request", "expected_urgency": "low"},
]

correct_category = 0
correct_urgency = 0

for item in test_data:
    pred = simple_extract(item["text"])
    if pred["category"] == item["expected_category"]:
        correct_category += 1
    if pred["urgency"] == item["expected_urgency"]:
        correct_urgency += 1

print(f"Category accuracy: {correct_category}/{len(test_data)}")
print(f"Urgency accuracy: {correct_urgency}/{len(test_data)}")

Category accuracy: 3/3
Urgency accuracy: 3/3


In [6]:
# This is a simple function-based interface for the demo
def run_copilot(ticket_text: str):
    result = simple_extract(ticket_text)
    if not validate_output(result):
        return {"error": "Validation failed"}
    return result

print(run_copilot("I was charged twice and I need a refund immediately"))

{'category': 'billing', 'urgency': 'high', 'summary': 'I was charged twice and I need a refund immediately', 'recommended_next_action': 'Review billing records and respond within 4 hours'}
